In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

First, let's load the dataset provided.

In [2]:
df = pd.read_csv('/content/customer_360_ml_workshop.csv')
display(df.head())

,CustomerID,Age,Region,TenureMonths,ContractType,InternetType,MonthlyUsageGB,MonthlyChargeEGP,NumServices,SupportCalls6M,LatePayments12M,PaperlessBilling,AutoPay,SatisfactionScore,Future12MRevenueEGP,Churn
0,CUST-00001,22,Alexandria,23,Month-to-month,DSL,317.9,580.48,5,5,0,Yes,Yes,5.0,6287.08,Yes
1,CUST-00002,59,Upper Egypt,9,Month-to-month,4G/5G,359.3,434.36,1,1,1,No,Yes,4.0,4619.91,No
2,CUST-00003,52,Giza,21,Month-to-month,DSL,176.9,391.35,3,5,1,Yes,Yes,5.0,4512.25,No
3,CUST-00004,41,Giza,12,Month-to-month,DSL,312.5,420.40,3,2,1,No,Yes,5.0,3809.65,No
4,CUST-00005,40,Delta,21,Month-to-month,4G/5G,191.6,462.73,3,4,0,No,Yes,3.0,4731.09,No


Now, I'll create a function `create_preprocessing_pipeline` that generates a `ColumnTransformer` based on the input DataFrame's column types, ensuring the specified imputation, scaling, and encoding steps are included. This pipeline will be reusable and avoid fitting on the entire dataset before train/test splitting.

In [5]:
def create_preprocessing_pipeline(dataframe):
    # Identify numerical and categorical columns
    numerical_cols = dataframe.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = dataframe.select_dtypes(include=['object', 'category']).columns.tolist()

    # Preprocessing for numerical features
    numerical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Preprocessing for categorical features
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # Ensure dense output
    ])

    # Create a column transformer to apply different transformations to different columns
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_cols),
            ('cat', categorical_transformer, categorical_cols)
        ],
        remainder='passthrough' # Keep other columns not specified (e.g., datetime, boolean, etc.)
    )

    return preprocessor

# Demonstrate the pipeline creation and application
preprocessor = create_preprocessing_pipeline(df)

# To see the output columns after transformation, we need to fit and transform a sample
# Note: In a real scenario, you would fit on training data and transform train/test data separately.
# For demonstration, we'll transform the whole dataframe.

# Fit and transform the data (for demonstration purposes)
X_processed = preprocessor.fit_transform(df)

# To get the feature names after one-hot encoding, we need to access them from the fitted onehot encoder
feature_names = preprocessor.get_feature_names_out()

# Convert the processed array back to a DataFrame for better inspection
df_processed = pd.DataFrame(X_processed, columns=feature_names)

display(df_processed.head())
print(f"Shape of original DataFrame: {df.shape}")
print(f"Shape of preprocessed DataFrame: {df_processed.shape}")

,num__Age,num__TenureMonths,num__MonthlyUsageGB,num__MonthlyChargeEGP,num__NumServices,num__SupportCalls6M,num__LatePayments12M,num__SatisfactionScore,num__Future12MRevenueEGP,cat__CustomerID_CUST-00001,...,cat__ContractType_Two year,cat__InternetType_4G/5G,cat__InternetType_DSL,cat__InternetType_Fiber,cat__PaperlessBilling_No,cat__PaperlessBilling_Yes,cat__AutoPay_No,cat__AutoPay_Yes,cat__Churn_No,cat__Churn_Yes
0,-1.437566,-0.153766,0.995290,0.786395,0.866496,1.858144,-1.042372,1.494735,0.625784,1.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
1,0.973838,-1.001070,1.464579,-0.447393,-1.467500,-0.802365,-0.104143,0.500675,-0.500970,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
2,0.517626,-0.274809,-0.603015,-0.810555,-0.300502,1.858144,-0.104143,1.494735,-0.573732,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
3,-0.199278,-0.819505,0.934078,-0.565267,-0.300502,-0.137238,-0.104143,1.494735,-1.048583,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
4,-0.264451,-0.274809,-0.436383,-0.207847,-0.300502,1.193017,-1.042372,-0.493385,-0.425829,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0


Shape of original DataFrame: (3000, 16)
Shape of preprocessed DataFrame: (3000, 3026)
